<font size=10>**TASK 3 - ??**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Project**: *Straining the great southern Melting Pot*

**Group 8**
- Beatriz Marques 20231605
- David Carrilho 20231693
- Duarte Fernandes 20231619
- Filipe Caçador 20231707
- Mariana Calais-Pedro 20231641

*«notebook description»*

**Question**: *??*

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a> 
- [1. Imports](#1)
- [2. Data](#2)
- [3. Named Entity Rcognition](#3)
    - [3.1 Specific Data Preparation](#3_1)
    - [3.2 Model Implementation](#3_2)
    - [3.3 Model Evaluation](#3_3)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [39]:
import sys
import os
import pandas as pd
import spacy
from spacy import displacy
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from my_utils import *
from visualizations import *
from general_preprocessing import *

# <font color='#BFD72F' size=6>**2. Data**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [3]:
dataset_original = load_dataset('../data/atlanta_restaurant_slice_2023.csv')

In [4]:
dataset_original.head()

,title,categoryName,website,url,reviewsCount,stars,text
0,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,"One word amazing!! The red fish, halibut, fr..."
1,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,First time here and the food is great and the ...
2,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,I recently had the pleasure of dining at Optim...
3,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,Beautiful atmosphere and delicious food. All o...
4,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,We had a wonderful dinner at the Optimist. Our...


In [5]:
dataset_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53566 entries, 0 to 53565
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         53566 non-null  object 
 1   categoryName  53566 non-null  object 
 2   website       50600 non-null  object 
 3   url           53566 non-null  object 
 4   reviewsCount  53566 non-null  int64  
 5   stars         53566 non-null  float64
 6   text          53566 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 2.9+ MB


| 🏷️ **Column Name** | 📝 **Description** |
|:-------------------|:-------------------|
|**title** | Name of the restaurant |
|**categoryName** | Labels that describe the restaurant's cuisine type |
|**website** | URL of the restaurant's webpage |
|**url** | URL of the restaurant's Google Maps page |
|**reviewsCount** | Total number of reviews for the restaurant at the time of scraping |
|**stars** | Customer rating (1 to 5) |
|**text** | Text of the review |

In [6]:
dataset = dataset_original.copy()

# <font color='#BFD72F' size=6>**3. Named Entity Recognition**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.1 Specific Data Preparation</font> <a class="anchor" id="3_1"></a>
  
[Back to TOC](#toc)

In [14]:
dataset["pos_list"] =\
      dataset["text"].map(lambda content : main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            pos_tags_list='pos_list'
                                                            ))
dataset["tokens"] =\
      dataset["text"].map(lambda content : main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            tokenized_output=True
                                                            ))


In [15]:
dataset

,title,categoryName,website,url,reviewsCount,stars,text,preproc_content,pos_list,tokens
0,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,"One word amazing!! The red fish, halibut, fr...","[CD, NN, NN, ., ., DT, JJ, NN, ,, NN, ,, VBN, ...","[CD, NN, NN, ., ., DT, JJ, NN, ,, NN, ,, VBN, ...","[One, word, amazing, !, !, The, red, fish, ,, ..."
1,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,First time here and the food is great and the ...,"[JJ, NN, RB, CC, DT, NN, VBZ, JJ, CC, DT, NN, ...","[JJ, NN, RB, CC, DT, NN, VBZ, JJ, CC, DT, NN, ...","[First, time, here, and, the, food, is, great,..."
2,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,I recently had the pleasure of dining at Optim...,"[PRP, RB, VBD, DT, NN, IN, VBG, IN, NNP, IN, N...","[PRP, RB, VBD, DT, NN, IN, VBG, IN, NNP, IN, N...","[I, recently, had, the, pleasure, of, dining, ..."
3,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,Beautiful atmosphere and delicious food. All o...,"[NNP, RB, CC, JJ, NN, ., DT, IN, DT, NN, NN, N...","[NNP, RB, CC, JJ, NN, ., DT, IN, DT, NN, NN, N...","[Beautiful, atmosphere, and, delicious, food, ..."
4,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,We had a wonderful dinner at the Optimist. Our...,"[PRP, VBD, DT, JJ, NN, IN, DT, NNP, ., PRP$, N...","[PRP, VBD, DT, JJ, NN, IN, DT, NNP, ., PRP$, N...","[We, had, a, wonderful, dinner, at, the, Optim..."
...,...,...,...,...,...,...,...,...,...,...
53561,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Friday night dinner was Chicken Francaise. Jor...,"[NNP, NN, NN, VBD, NNP, NNP, ., NNP, VBD, DT, ...","[NNP, NN, NN, VBD, NNP, NNP, ., NNP, VBD, DT, ...","[Friday, night, dinner, was, Chicken, Francais..."
53562,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Great dinner.... Yay Jordan!!!!!!,"[NNP, NNP, NNP, ., ., ., ., ., .]","[NNP, NNP, NNP, ., ., ., ., ., .]","[Great, Yay, Jordan, !, !, !, !, !, !]"
53563,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Jordan was our server and he was fantastic! Gr...,"[NNP, VBD, PRP$, NN, CC, PRP, VBD, JJ, ., JJ, ...","[NNP, VBD, PRP$, NN, CC, PRP, VBD, JJ, ., JJ, ...","[Jordan, was, our, server, and, he, was, fanta..."
53564,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Jordan was an amazing server! Great delicious...,"[NNP, VBD, DT, JJ, NN, ., NNP, JJ, NN, CC, NN,...","[NNP, VBD, DT, JJ, NN, ., NNP, JJ, NN, CC, NN,...","[Jordan, was, an, amazing, server, !, Great, d..."


In [17]:
dataset["features"] = dataset.apply(lambda row : sent2features(row["tokens"], row["pos_list"]), axis=1)
dataset["features"].sample(1).iloc[0]

[{'bias': 1.0,
  'word.lower()': 'food',
  'word[-3:]': 'ood',
  'word[-2:]': 'od',
  'word.isupper()': False,
  'word.istitle()': True,
  'word.isdigit()': False,
  'postag': 'NN',
  'postag[:2]': 'NN',
  'BOS': True,
  '+1:word.lower()': 'is',
  '+1:word.istitle()': False,
  '+1:word.isupper()': False,
  '+1:postag': 'VBZ',
  '+1:postag[:2]': 'VB'},
 {'bias': 1.0,
  'word.lower()': 'is',
  'word[-3:]': 'is',
  'word[-2:]': 'is',
  'word.isupper()': False,
  'word.istitle()': False,
  'word.isdigit()': False,
  'postag': 'VBZ',
  'postag[:2]': 'VB',
  '-1:word.lower()': 'food',
  '-1:word.istitle()': True,
  '-1:word.isupper()': False,
  '-1:postag': 'NN',
  '-1:postag[:2]': 'NN',
  '+1:word.lower()': 'excellent',
  '+1:word.istitle()': False,
  '+1:word.isupper()': False,
  '+1:postag': 'JJ',
  '+1:postag[:2]': 'JJ'},
 {'bias': 1.0,
  'word.lower()': 'excellent',
  'word[-3:]': 'ent',
  'word[-2:]': 'nt',
  'word.isupper()': False,
  'word.istitle()': False,
  'word.isdigit()': False

In [22]:
# Expand rows so each token becomes its own row

dataset["ner_labels"] = dataset.apply(
    lambda row: align_bio(nlp(row["text"]), row["tokens"]),
    axis=1
)

In [24]:
dataset

,title,categoryName,website,url,reviewsCount,stars,text,preproc_content,pos_list,tokens,features,ner_labels
0,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,"One word amazing!! The red fish, halibut, fr...","[CD, NN, NN, ., ., DT, JJ, NN, ,, NN, ,, VBN, ...","[CD, NN, NN, ., ., DT, JJ, NN, ,, NN, ,, VBN, ...","[One, word, amazing, !, !, The, red, fish, ,, ...","[{'bias': 1.0, 'word.lower()': 'one', 'word[-3...","[B-CARDINAL, O, O, O, O, O, O, O, O, O, O, O, ..."
1,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,First time here and the food is great and the ...,"[JJ, NN, RB, CC, DT, NN, VBZ, JJ, CC, DT, NN, ...","[JJ, NN, RB, CC, DT, NN, VBZ, JJ, CC, DT, NN, ...","[First, time, here, and, the, food, is, great,...","[{'bias': 1.0, 'word.lower()': 'first', 'word[...","[B-ORDINAL, O, O, O, O, O, O, O, O, O, O, O, O..."
2,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,I recently had the pleasure of dining at Optim...,"[PRP, RB, VBD, DT, NN, IN, VBG, IN, NNP, IN, N...","[PRP, RB, VBD, DT, NN, IN, VBG, IN, NNP, IN, N...","[I, recently, had, the, pleasure, of, dining, ...","[{'bias': 1.0, 'word.lower()': 'i', 'word[-3:]...","[O, O, O, O, O, O, O, O, B-PERSON, O, B-GPE, O..."
3,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,Beautiful atmosphere and delicious food. All o...,"[NNP, RB, CC, JJ, NN, ., DT, IN, DT, NN, NN, N...","[NNP, RB, CC, JJ, NN, ., DT, IN, DT, NN, NN, N...","[Beautiful, atmosphere, and, delicious, food, ...","[{'bias': 1.0, 'word.lower()': 'beautiful', 'w...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,We had a wonderful dinner at the Optimist. Our...,"[PRP, VBD, DT, JJ, NN, IN, DT, NNP, ., PRP$, N...","[PRP, VBD, DT, JJ, NN, IN, DT, NNP, ., PRP$, N...","[We, had, a, wonderful, dinner, at, the, Optim...","[{'bias': 1.0, 'word.lower()': 'we', 'word[-3:...","[O, O, O, O, O, O, O, B-NORP, O, O, O, O, B-CA..."
...,...,...,...,...,...,...,...,...,...,...,...,...
53561,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Friday night dinner was Chicken Francaise. Jor...,"[NNP, NN, NN, VBD, NNP, NNP, ., NNP, VBD, DT, ...","[NNP, NN, NN, VBD, NNP, NNP, ., NNP, VBD, DT, ...","[Friday, night, dinner, was, Chicken, Francais...","[{'bias': 1.0, 'word.lower()': 'friday', 'word...","[B-TIME, I-TIME, O, O, B-PERSON, I-PERSON, O, ..."
53562,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Great dinner.... Yay Jordan!!!!!!,"[NNP, NNP, NNP, ., ., ., ., ., .]","[NNP, NNP, NNP, ., ., ., ., ., .]","[Great, Yay, Jordan, !, !, !, !, !, !]","[{'bias': 1.0, 'word.lower()': 'great', 'word[...","[O, O, B-PERSON, O, O, O, O, O, O]"
53563,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Jordan was our server and he was fantastic! Gr...,"[NNP, VBD, PRP$, NN, CC, PRP, VBD, JJ, ., JJ, ...","[NNP, VBD, PRP$, NN, CC, PRP, VBD, JJ, ., JJ, ...","[Jordan, was, our, server, and, he, was, fanta...","[{'bias': 1.0, 'word.lower()': 'jordan', 'word...","[B-PERSON, O, O, O, O, O, O, O, O, O, O, O]"
53564,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Jordan was an amazing server! Great delicious...,"[NNP, VBD, DT, JJ, NN, ., NNP, JJ, NN, CC, NN,...","[NNP, VBD, DT, JJ, NN, ., NNP, JJ, NN, CC, NN,...","[Jordan, was, an, amazing, ser

In [34]:
nlp = spacy.load("en_core_web_sm")
equivalence_table = {"PERSON":"-per",
                     "NORP":"-grp",
                     "FAC":"-fac",
                     "ORG":"-org",
                     "GPE":"-gpe",
                     "LOC":"-geo",
                     "DATE":"-date",
                     "TIME":"-tim",
                     "WORK_OF_ART":"-art",
                     "LAW":"-law",
                     "LANGUAGE":"-lang",
                     "EVENT":"eve",
                     "PRODUCT":"-prod",
                     "PERCENT":"-pct",
                     "MONEY":"-mon",
                     "QUANTITY":"-qty",
                     "CARDINAL":"-card",
                     "ORDINAL":"-ord",
                     "":""
                     }

dataset["ner_labels_custom"] = dataset.apply(
    lambda row: align_bio_to_custom_tokens(
        row["text"], 
        row["tokens"],
        nlp,
        equivalence_table
    ),
    axis=1
)

In [35]:
dataset

,title,categoryName,website,url,reviewsCount,stars,text,preproc_content,pos_list,tokens,features,ner_labels,ner_labels_custom
0,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,"One word amazing!! The red fish, halibut, fr...","[CD, NN, NN, ., ., DT, JJ, NN, ,, NN, ,, VBN, ...","[CD, NN, NN, ., ., DT, JJ, NN, ,, NN, ,, VBN, ...","[One, word, amazing, !, !, The, red, fish, ,, ...","[{'bias': 1.0, 'word.lower()': 'one', 'word[-3...","[B-CARDINAL, O, O, O, O, O, O, O, O, O, O, O, ...","[B-card, O, O, O, O, O, O, O, O, O, O, O, O, O..."
1,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,First time here and the food is great and the ...,"[JJ, NN, RB, CC, DT, NN, VBZ, JJ, CC, DT, NN, ...","[JJ, NN, RB, CC, DT, NN, VBZ, JJ, CC, DT, NN, ...","[First, time, here, and, the, food, is, great,...","[{'bias': 1.0, 'word.lower()': 'first', 'word[...","[B-ORDINAL, O, O, O, O, O, O, O, O, O, O, O, O...","[B-ord, O, O, O, O, O, O, O, O, O, O, O, O, O]"
2,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,I recently had the pleasure of dining at Optim...,"[PRP, RB, VBD, DT, NN, IN, VBG, IN, NNP, IN, N...","[PRP, RB, VBD, DT, NN, IN, VBG, IN, NNP, IN, N...","[I, recently, had, the, pleasure, of, dining, ...","[{'bias': 1.0, 'word.lower()': 'i', 'word[-3:]...","[O, O, O, O, O, O, O, O, B-PERSON, O, B-GPE, O...","[O, O, O, O, O, O, O, O, B-per, O, B-gpe, O, B..."
3,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,Beautiful atmosphere and delicious food. All o...,"[NNP, RB, CC, JJ, NN, ., DT, IN, DT, NN, NN, N...","[NNP, RB, CC, JJ, NN, ., DT, IN, DT, NN, NN, N...","[Beautiful, atmosphere, and, delicious, food, ...","[{'bias': 1.0, 'word.lower()': 'beautiful', 'w...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,We had a wonderful dinner at the Optimist. Our...,"[PRP, VBD, DT, JJ, NN, IN, DT, NNP, ., PRP$, N...","[PRP, VBD, DT, JJ, NN, IN, DT, NNP, ., PRP$, N...","[We, had, a, wonderful, dinner, at, the, Optim...","[{'bias': 1.0, 'word.lower()': 'we', 'word[-3:...","[O, O, O, O, O, O, O, B-NORP, O, O, O, O, B-CA...","[O, O, O, O, O, O, O, B-grp, O, O, O, O, B-car..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
53561,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Friday night dinner was Chicken Francaise. Jor...,"[NNP, NN, NN, VBD, NNP, NNP, ., NNP, VBD, DT, ...","[NNP, NN, NN, VBD, NNP, NNP, ., NNP, VBD, DT, ...","[Friday, night, dinner, was, Chicken, Francais...","[{'bias': 1.0, 'word.lower()': 'friday', 'word...","[B-TIME, I-TIME, O, O, B-PERSON, I-PERSON, O, ...","[B-tim, I-tim, O, O, B-per, I-per, O, B-per, O..."
53562,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Great dinner.... Yay Jordan!!!!!!,"[NNP, NNP, NNP, ., ., ., ., ., .]","[NNP, NNP, NNP, ., ., ., ., ., .]","[Great, Yay, Jordan, !, !, !, !, !, !]","[{'bias': 1.0, 'word.lower()': 'great', 'word[...","[O, O, B-PERSON, O, O, O, O, O, O]","[O, O, B-per, O, O, O, O, O, O]"
53563,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Jordan was our server and he was fantastic! Gr...,"[NNP, VBD, PRP$, NN, CC, PRP, VBD, JJ, ., JJ, ...","[NNP, VBD, PRP$, NN, CC, PRP, VBD, JJ, ., JJ, ...","[Jordan, was, our, server, and, he, was, fanta...","[{'bias': 1.0, 'word.lower()': 'jordan', 'word...","[B-PERSON, O,

## <font color='#BFD72F' size=6>3.2 Model Implementation</font> <a class="anchor" id="3_2"></a>
  
[Back to TOC](#toc)

In [41]:
#Create a quick (q1) train-test split that we can use to test our classifier
X_train_q1, X_test_q1, y_train_q1, y_test_q1 = train_test_split(dataset["features"], dataset["ner_labels_custom"],
                                                    test_size=0.2, random_state=39)

train_index_q1 = list(X_train_q1.index)
test_index_q1 = list(X_test_q1.index)

crf = sklearn_crfsuite.CRF(algorithm='lbfgs', 
                           c1=0.1, 
                           c2=0.1, 
                           max_iterations=100, 
                           all_possible_transitions=True,
                           verbose=True 
                           )
try:
    crf.fit(X_train_q1, y_train_q1)
except AttributeError:
    pass

X_test_tokens_q1 = dataset["tokens"].loc[test_index_q1]

y_pred_q1 = crf.predict(X_test_q1)



loading training data to CRFsuite: 100%|██████████| 42852/42852 [00:30<00:00, 1403.71it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 137038
Seconds required: 3.718

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 100
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=14.54 loss=3408241.87 active=136259 feature_norm=1.00
Iter 2   time=14.82 loss=2161165.49 active=136015 feature_norm=14.90
Iter 3   time=7.14  loss=2015023.96 active=124968 feature_norm=14.04
Iter 4   time=117.89 loss=462174.63 active=84140 feature_norm=5.89
Iter 5   time=56.56 loss=385032.07 active=97763 feature_norm=5.88
Iter 6   time=71.23 loss=382926.62 active=101882 feature_norm=5.68
Iter 7   time=56.49 loss=372977.97 active=107746 feature_norm=5.67
Iter 8   time=63.33 loss=371057.44 active=108124 feature_norm=5.71
Iter 9   time=56.11 loss=368005.49 active=104890 feature_n

## <font color='#BFD72F' size=6>3.3 Model Evaluation</font> <a class="anchor" id="3_3"></a>
  
[Back to TOC](#toc)

In [42]:
labels = list(crf.classes_)
labels.remove("O")

round(metrics.flat_f1_score(y_test_q1, y_pred_q1,average='weighted', labels=labels), 3)

0.722

In [43]:
sorted_labels = sorted(
    labels,
    key=lambda name: (name[1:], name[0])
)

print(sklearn_crfsuite.metrics.flat_classification_report(y_test_q1, y_pred_q1, labels=sorted_labels, digits=3))

              precision    recall  f1-score   support

       B-art      0.630     0.193     0.296        88
       I-art      0.615     0.240     0.345       100
      B-card      0.823     0.861     0.841      2122
      I-card      0.821     0.719     0.767       256
      B-date      0.847     0.826     0.836      1389
      I-date      0.850     0.824     0.837      1224
       B-fac      0.533     0.367     0.435       109
       I-fac      0.520     0.387     0.444       137
       B-geo      0.480     0.379     0.424        95
       I-geo      0.214     0.154     0.179        39
       B-gpe      0.796     0.704     0.747       998
       I-gpe      0.801     0.717     0.757       180
       B-grp      0.812     0.787     0.800       968
       I-grp      0.818     0.514     0.632        35
      B-lang      0.500     0.615     0.552        13
       B-law      0.600     0.300     0.400        20
       I-law      0.471     0.250     0.327        32
       B-mon      0.826    